# 0. 원천 데이터 → els3_dataset 생성 (base)

raw DART(`AUTO_CALL`/`SCHD_INFO`/`UDLY_INFO`) + `data/cache`(기초자산 일별종가 `px_*`, KRW 금리곡선 `krw_curve`)로부터 **3-star STEP KRW ELS**의 학습 base 데이터셋을 직접 재유도해 `data/els3_dataset.parquet` 로 저장한다.

- **branch(시장상태)**: 바스켓 변동성 `sig1..3`/`sig_eff` (180일 역사), 상관 `rho12/13/23` (180일 역사), KRW Nelson-Siegel 수익률곡선 `u0..u9`/`r`/`curve_*`.
- **계약/보조**: `B`/`coupon`/`tenor`/`nobs`/`cpn_spread`/`b_over_k`/`stepdown`/`mom6m` + 발행메타(issuer/risk/…), `fair`(공정가치), 인과적 `recent_mktvol`.

**이 단계엔 MC 이론가를 계산하지 않는다** — `mc`·`recent_margin` 은 `1_MC_recompute` 에서 추가하며, 학습 CSV 2종(`ml`/`deeponet`)도 mc 가 붙은 뒤 거기서 생성한다. (이 노트북은 raw→base 검증용)

In [1]:
# ===== raw + cache → els3_dataset base 재유도 =====
# module.build_source: exp15 로직을 현행 정의(180일 역사 vol/corr, KRW NS 공섬)로 이식. MC 미포함.
from module import build_source
df = build_source.build_source()
print("\ncolumns:", len(df.columns))
print(list(df.columns))

candidate 3-star STEP KRW: 48192
built 23151 products in 85s
saved c:\Users\Juhwankim\Documents\Github\PI-DeepONet-ELS\data\els3_dataset.parquet rows 23151

columns: 51
['item', 'sig1', 'sig2', 'sig3', 'rho12', 'rho13', 'rho23', 'sig_mean', 'sig_max', 'sig_min', 'rho', 'sig_eff', 'cpn_spread', 'b_over_k', 'stepdown', 'mom6m', 'isu_ord', 'r', 'B', 'Kfirst', 'K', 'coupon', 'tenor', 'nobs', 'fair', 'issuer', 'risk', 'ptype', 'rdmp', 'imonth', 'amt', 'sbrt', 'dvrt', 'prcp', 'kigrc', 'iyear', 'subdays', 'u0', 'u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8', 'u9', 'curve_level', 'curve_slope', 'curve_curv', 'recent_mktvol']


## base 검증

MC 전 base 상태 확인: branch 피처 완전성(NaN 0), 범위, 발행기간. `mc`·`recent_margin` 은 아직 없어야 정상(1_MC_recompute에서 추가).

In [2]:
import pandas as pd, numpy as np
from util import file_manager as fm

d = pd.read_parquet(fm.source())
branch = ["sig1", "sig2", "sig3", "rho12", "rho13", "rho23", "sig_eff",
          "u0", "u9", "r", "curve_level", "curve_slope", "curve_curv", "recent_mktvol"]
print(f"source -> {fm.source().relative_to(fm.ROOT)} | rows {len(d)} | cols {len(d.columns)}")
print(f"branch 피처 present: {all(c in d.columns for c in branch)} | NaN: {int(d[branch + ['fair']].isna().sum().sum())}")
print(f"has mc? {'mc' in d.columns} | has recent_margin? {'recent_margin' in d.columns}  (둘 다 False 가 정상)")
print(f"fair [{d.fair.min():.3f}, {d.fair.max():.3f}] | tenor [{d.tenor.min():.2f}, {d.tenor.max():.2f}] | "
      f"sig_eff 평균 {d.sig_eff.mean():.3f}")
yr = pd.to_datetime(d.isu_ord.map(lambda o: pd.Timestamp.fromordinal(int(o))))
print(f"발행기간 {yr.min().date()} ~ {yr.max().date()}")
display(d[branch + ["fair", "coupon", "tenor"]].describe().round(4))

source -> data\els3_dataset.parquet | rows 23151 | cols 51
branch 피처 present: True | NaN: 0
has mc? False | has recent_margin? False  (둘 다 False 가 정상)
fair [0.704, 1.050] | tenor [0.99, 5.00] | sig_eff 평균 0.246
발행기간 2015-01-05 ~ 2026-05-08


,sig1,sig2,sig3,rho12,rho13,rho23,sig_eff,u0,u9,r,curve_level,curve_slope,curve_curv,recent_mktvol,fair,coupon,tenor
count,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000,23151.0000
mean,0.1488,0.1868,0.2393,0.2006,0.3068,0.5354,0.2457,0.0191,0.0243,0.0283,0.0253,0.0052,0.0118,0.1907,0.9311,0.0652,2.9948
std,0.0448,0.0572,0.0741,0.1274,0.1259,0.1227,0.0668,0.0091,0.0072,0.0102,0.0090,0.0052,0.0101,0.0465,0.0493,0.0245,0.1841
min,0.0665,0.0958,0.1023,-0.1556,-0.0987,0.0566,0.1185,0.0063,0.0125,0.0110,0.0124,-0.0058,-0.0058,0.1065,0.7041,0.0000,0.9938
25%,0.1217,0.1435,0.1932,0.1117,0.2213,0.4683,0.2022,0.0139,0.0191,0.0225,0.0200,0.0012,0.0063,0.1656,0.8988,0.0500,2.9925
50%,0.1362,0.1783,0.2259,0.1967,0.2893,0.5393,0.2341,0.0165,0.0229,0.0258,0.0230,0.0054,0.0091,0.1806,0.9357,0.0620,2.9979
75%,0.1678,0.2137,0.2758,0.2912,0.3945,0.6085,0.2784,0.0251,0.0283,0.0304,0.0277,0.0092,0.0152,0.2127,0.9688,0.0780,3.0007
max,0.4034,0.5522,0.6983,0.7705,0.8099,0.9990,0.6858,0.0402,0.0427,0.0755,0.0604,0.0165,0.0733,0.3457,1.0497,0.3000,4.9993
